# 이란-미국 분쟁 뉴스 헤드라인 키워드 빈도와 유가 상관관계 분석 — v7

> **v7 변경점**: Figma 피치 덱과 동일한 cyan/amber 다크 테마 적용.
> 모든 시각화 (시계열, 히트맵, 산점도, 비교 차트)에 동일한 색상 시스템 사용.


**기간:** 2026-01-01 ~ 2026-05-20 (KST 기준)

**뉴스 소스:** Google News RSS (한국 / 미국 분리 수집)

**유가 벤치마크:** Brent (BZ=F), WTI (CL=F)

**분석 흐름:**
1. 뉴스 헤드라인 수집 (한국·미국)
2. 유가 데이터 수집
3. 일별 키워드 빈도 매트릭스 생성
4. 시차(lag) 상관분석
5. 시각화 (시계열 / 히트맵 / 워드클라우드 / 비교 패널)


## 0. 환경 설정 및 라이브러리


In [ ]:
# 설치가 필요하면 한 번만 실행
# !pip install -r requirements.txt


In [ ]:
import re
import time
import urllib.parse
from datetime import datetime, timedelta
from pathlib import Path

import feedparser
import pandas as pd
import numpy as np
import pytz
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats

# 한국어 폰트 자동 탐색 + 등록 (TTF 경로 직접 지정 방식 — 가장 안정적)
import matplotlib.font_manager as _fm

def setup_korean_font():
    import os
    candidate_paths = [
        os.path.expanduser("~/.local/share/fonts/malgun.ttf"),
        os.path.expanduser("~/.local/share/fonts/NanumGothic.ttf"),
        "/mnt/c/Windows/Fonts/malgun.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/Library/Fonts/AppleGothic.ttf",
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            try:
                _fm.fontManager.addfont(p)
                prop = _fm.FontProperties(fname=p)
                family = prop.get_name()
                mpl.rcParams["font.family"] = family
                # findfont LRU 캐시 무효화 (matplotlib 3.x에서 addfont 후 필요)
                try:
                    _fm.findfont.cache_clear()
                except Exception:
                    pass
                print(f"[font] using: {family}  (from {p})")
                return prop  # FontProperties 객체를 반환 — 명시 fontproperties용
            except Exception as e:
                print(f"[font] addfont 실패 {p}: {e}")
                continue
    print("[font] 한글 폰트를 찾지 못함 — 한글이 깨질 수 있음")
    return None

KOR_FP = setup_korean_font()  # FontProperties 객체

def apply_korean_font(ax):
    """플롯의 모든 텍스트 요소에 한국어 폰트 강제 적용 — findfont 캐시 회피용."""
    if KOR_FP is None:
        return
    for txt in [ax.title, ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
        txt.set_fontproperties(KOR_FP)
    leg = ax.get_legend()
    if leg is not None:
        for t in leg.get_texts():
            t.set_fontproperties(KOR_FP)
mpl.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

KST = pytz.timezone("Asia/Seoul")
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

# 캐시 토글: True면 data/*.csv가 있을 때 RSS 재호출 없이 로드 (v3 빠른 실행용)
USE_CACHE = True

print("setup complete")


## 0.5 v7 — Design System (cyan accent · dark theme)

V7는 Figma 피치 덱과 동일한 색상 시스템을 사용한다.
- **다크 배경** `#0A0A0B` + **off-white 텍스트** `#FAFAFA`
- **단일 액센트 cyan** `#06B6D4` (주요 키워드, 강조)
- **amber** `#F59E0B` (유가 — Brent), **violet** `#A855F7` (유가 — WTI)
- **커스텀 colormap**: `CMAP_DIV` (amber → black → cyan, 다이버징 상관계수용), `CMAP_SEQ` (black → cyan, 시퀀셜)


In [ ]:
# ============================================================
# v7 — Design System (cyan accent · dark theme)
# Figma pitch deck과 동일한 색상 토큰 + 커스텀 colormap
# ============================================================
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Color tokens
COLOR = {
    "bg":      "#0A0A0B",   # warm pure black
    "fg":      "#FAFAFA",   # off-white
    "g_400":   "#A1A1AA",   # zinc-400 (axis ticks)
    "g_500":   "#71717A",   # zinc-500 (secondary)
    "g_700":   "#3F3F46",   # zinc-700 (spines)
    "g_800":   "#27272A",   # zinc-800 (grid)
    "cyan":    "#06B6D4",   # primary accent
    "cyan_lt": "#67E8F9",   # bright cyan (highlight)
    "amber":   "#F59E0B",   # Brent / strings
    "violet":  "#A855F7",   # WTI / numbers
}

# Diverging colormap — amber (neg) → black (zero) → cyan (pos)
CMAP_DIV = LinearSegmentedColormap.from_list(
    "cps_div",
    ["#F59E0B", "#B07005", "#3F2A00", "#0A0A0B", "#003E4A", "#06B6D4", "#67E8F9"],
    N=256
)

# Sequential colormap — black → cyan
CMAP_SEQ = LinearSegmentedColormap.from_list(
    "cps_seq",
    ["#0A0A0B", "#0A2D33", "#06B6D4", "#67E8F9"],
    N=256
)

# Seaborn 베이스라인 (style만 — rcParams는 아래에서 덮어쓰기)
sns.set_style("ticks")

# matplotlib rcParams — winning settings
plt.rcParams.update({
    "figure.facecolor":   COLOR["bg"],
    "axes.facecolor":     COLOR["bg"],
    "axes.edgecolor":     COLOR["g_700"],
    "axes.labelcolor":    COLOR["fg"],
    "axes.titlecolor":    COLOR["fg"],
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "text.color":         COLOR["fg"],
    "xtick.color":        COLOR["g_400"],
    "ytick.color":        COLOR["g_400"],
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "grid.color":         COLOR["g_800"],
    "grid.linewidth":     0.5,
    "grid.linestyle":     "-",
    "axes.grid":          True,
    "axes.axisbelow":     True,
    "legend.facecolor":   COLOR["bg"],
    "legend.edgecolor":   COLOR["g_700"],
    "legend.labelcolor":  COLOR["fg"],
    "legend.fontsize":    10,
    "savefig.facecolor":  COLOR["bg"],
    "savefig.edgecolor":  COLOR["bg"],
    "savefig.dpi":        150,
    "patch.facecolor":    COLOR["bg"],
    "patch.edgecolor":    COLOR["g_700"],
})

# CRITICAL: IPython InlineBackend는 savefig 호출 시 자체 facecolor를 강제함.
# print_figure_kwargs를 통해 dark facecolor를 전달해야 inline render에서도 다크 유지.
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic(
            "config",
            f"InlineBackend.print_figure_kwargs = {{'facecolor': '{COLOR['bg']}', 'edgecolor': '{COLOR['bg']}', 'bbox_inches': 'tight'}}"
        )
        # 또한 rc도 클리어 (선택적 추가 보호)
        ip.run_line_magic("config", "InlineBackend.rc = {}")
except Exception as e:
    print(f"[warn] IPython config skip: {e}")

print("v7 design system loaded — cyan accent · dark theme")
print(f"  CMAP_DIV: amber → black → cyan")
print(f"  CMAP_SEQ: black → cyan")


## 1. 기간 및 키워드 사전

- 분석 기간은 KST 기준 2026-01-01 ~ 2026-05-20
- 별칭(alias)이 여러 개인 키워드는 **하나라도 헤드라인에 등장하면 +1**로 카운트


In [ ]:
START_DATE = "2026-01-01"
END_DATE   = "2026-05-20"

# 한국 뉴스용 키워드 사전 (표시명 -> 매칭 별칭 리스트)
KEYWORDS_KR = {
    "호르무즈 해협":         ["호르무즈 해협", "호르무즈"],
    "혁명수비대(IRGC)":      ["혁명수비대", "IRGC"],
    "시아 벨트":             ["시아 벨트", "시아벨트"],
    "테헤란 정권":           ["테헤란 정권", "테헤란"],
    "샤헤드 드론":           ["샤헤드 드론", "샤헤드"],
    "대리세력 네트워크":     ["대리세력", "프록시"],
    "핵협상(JCPOA)":         ["핵협상", "JCPOA"],
    "중동 저항축":           ["저항축", "Axis of Resistance"],
    "헤즈볼라 연계":         ["헤즈볼라"],
    "페르시아만 긴장":       ["페르시아만"],
    "제재 경제":             ["제재"],
    "비대칭 전력 전략":      ["비대칭 전력", "비대칭전"],
    "시아파 영향권":         ["시아파"],
    "솔레이마니 사망 사건":  ["솔레이마니"],
}

# 미국 뉴스용 키워드 사전 (영문)
KEYWORDS_US = {
    "Strait of Hormuz":      ["Strait of Hormuz", "Hormuz"],
    "IRGC":                  ["IRGC", "Revolutionary Guard"],
    "Shia Belt":             ["Shia belt", "Shi'a belt", "Shia crescent"],
    "Tehran Regime":         ["Tehran regime", "Tehran"],
    "Shahed Drone":          ["Shahed drone", "Shahed"],
    "Proxy Network":         ["proxy network", "proxies", "proxy forces"],
    "JCPOA":                 ["JCPOA", "nuclear deal"],
    "Axis of Resistance":    ["Axis of Resistance"],
    "Hezbollah":             ["Hezbollah"],
    "Persian Gulf":          ["Persian Gulf"],
    "Sanctions":             ["sanctions"],
    "Asymmetric Warfare":    ["asymmetric warfare", "asymmetric"],
    "Shia Influence":        ["Shia influence", "Shiite", "Shia"],
    "Soleimani":             ["Soleimani"],
}

print(f"KR keywords: {len(KEYWORDS_KR)}, US keywords: {len(KEYWORDS_US)}")


## 2. Google News RSS로 뉴스 헤드라인 수집 (v2 — 윈도우 스플릿)

전략:
- **7일 윈도우 × 30개 키워드 쿼리** = 한쪽당 ~600쿼리 → RSS의 100건/쿼리 한계 우회
- `after:` / `before:` 좁은 기간 필터로 다양한 시점의 헤드라인 끌어오기
- 한국(`hl=ko&gl=KR`)과 미국(`hl=en-US&gl=US`)을 분리 수집
- 목표: **일별 ~500건** (best effort)


In [ ]:
def fetch_google_news_rss(query, lang="ko", country="KR", lang_full=None, sleep_sec=0.5):
    """Google News RSS에서 검색 결과를 가져온다."""
    if lang_full is None:
        lang_full = lang
    encoded = urllib.parse.quote(query)
    url = (
        f"https://news.google.com/rss/search?q={encoded}"
        f"&hl={lang_full}&gl={country}&ceid={country}:{lang}"
    )
    feed = feedparser.parse(url)
    items = []
    for entry in feed.entries:
        # source 태그 처리 (feedparser 버전에 따라 형태 다름)
        source_title = ""
        if hasattr(entry, "source"):
            src = entry.source
            if isinstance(src, dict):
                source_title = src.get("title", "")
            else:
                source_title = getattr(src, "title", "") or ""
        items.append({
            "title": entry.get("title", "").strip(),
            "link": entry.get("link", ""),
            "published_raw": entry.get("published", ""),
            "source": source_title,
            "query": query,
        })
    time.sleep(sleep_sec)
    return items


def date_windows(start_date, end_date, chunk_days=7):
    """기간을 작은 윈도우로 쪼개 RSS의 ~100건/쿼리 한계를 우회."""
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + pd.Timedelta(days=chunk_days), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows


def collect_news_windowed(queries, lang, country, lang_full,
                           start_date, end_date, chunk_days=7, sleep_sec=0.5):
    """날짜 윈도우 × 키워드 쿼리 매트릭스로 수집 후 중복 제거."""
    windows = date_windows(start_date, end_date, chunk_days=chunk_days)
    total = len(queries) * len(windows)
    print(f"  → {len(windows)} windows × {len(queries)} queries = {total} 쿼리")
    all_items = []
    cnt = 0
    for w_start, w_end in windows:
        for q in queries:
            cnt += 1
            full_q = f"{q} after:{w_start} before:{w_end}"
            try:
                items = fetch_google_news_rss(full_q, lang, country, lang_full,
                                              sleep_sec=sleep_sec)
                all_items.extend(items)
            except Exception as e:
                print(f"  [{cnt}/{total}] {q!r} {w_start} 실패: {e}")
            if cnt % 60 == 0:
                df_tmp = pd.DataFrame(all_items).drop_duplicates(subset=["title"])
                print(f"  [{cnt}/{total}]  raw={len(all_items)}, unique={len(df_tmp)}")
    df = pd.DataFrame(all_items)
    if df.empty:
        return df
    df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
    return df


In [ ]:
# 한국 뉴스 쿼리 — 30개로 확장 + 7일 윈도우 split (일별 500건 목표)
RSS_QUERIES_KR = [
    "이란 미국", "이란 분쟁", "이란 군사", "이란 핵", "이란 제재",
    "이란 공습", "이란 미사일", "이란 보복", "이란 공격", "이란 위기",
    "호르무즈", "혁명수비대", "IRGC", "헤즈볼라", "JCPOA",
    "핵협상", "솔레이마니", "샤헤드", "테헤란", "중동 정세",
    "중동 분쟁", "페르시아만", "시아파", "이스라엘 이란", "이란 전쟁",
    "트럼프 이란", "후티 이란", "예멘 이란", "이란 드론", "이란 협상",
]
print(f"한국 RSS 쿼리 수: {len(RSS_QUERIES_KR)}")


In [ ]:
CACHE_KR = DATA_DIR / "news_kr.csv"
if USE_CACHE and CACHE_KR.exists():
    df_kr_raw = pd.read_csv(CACHE_KR)
    print(f"📦 캐시 로드: {CACHE_KR.name} — {len(df_kr_raw)}건 (RSS 재호출 생략)")
else:
    print("=== 한국 뉴스 수집 시작 ===")
    df_kr_raw = collect_news_windowed(RSS_QUERIES_KR, lang="ko", country="KR",
                                      lang_full="ko",
                                      start_date=START_DATE, end_date=END_DATE,
                                      chunk_days=7, sleep_sec=0.5)
    print(f"한국 뉴스 (중복 제거 후): {len(df_kr_raw)}건")
df_kr_raw.head()


In [ ]:
# 미국 뉴스 쿼리 — 30개로 확장
RSS_QUERIES_US = [
    "Iran US", "Iran United States", "Iran conflict", "Iran military", "Iran nuclear",
    "Iran sanctions", "Iran strike", "Iran retaliation", "Iran attack", "Iran crisis",
    "Strait of Hormuz", "IRGC", "Revolutionary Guard", "Hezbollah", "JCPOA",
    "nuclear deal Iran", "Soleimani", "Shahed drone", "Tehran", "Middle East tensions",
    "Persian Gulf", "Shia militia", "Israel Iran", "Iran war", "Trump Iran",
    "Houthi Iran", "Yemen Iran", "Iran drone", "Iran threat", "Iran missile",
]
print(f"미국 RSS 쿼리 수: {len(RSS_QUERIES_US)}")


In [ ]:
CACHE_US = DATA_DIR / "news_us.csv"
if USE_CACHE and CACHE_US.exists():
    df_us_raw = pd.read_csv(CACHE_US)
    print(f"📦 캐시 로드: {CACHE_US.name} — {len(df_us_raw)}건 (RSS 재호출 생략)")
else:
    print("=== 미국 뉴스 수집 시작 ===")
    df_us_raw = collect_news_windowed(RSS_QUERIES_US, lang="en", country="US",
                                      lang_full="en-US",
                                      start_date=START_DATE, end_date=END_DATE,
                                      chunk_days=7, sleep_sec=0.5)
    print(f"미국 뉴스 (중복 제거 후): {len(df_us_raw)}건")
df_us_raw.head()


### 2-1. 타임스탬프 KST 통일 + 날짜 컬럼 생성


In [ ]:
def normalize_timestamps(df, start_date, end_date):
    """published_raw -> KST datetime + date 컬럼, 기간 외 데이터 제거."""
    if df.empty:
        return df
    df = df.copy()
    # RFC 822 포맷 명시 (pandas 3.0이 자동추론 시 절반 이상 누락하는 이슈 회피)
    df["published_utc"] = pd.to_datetime(
        df["published_raw"], format="%a, %d %b %Y %H:%M:%S %Z",
        utc=True, errors="coerce",
    )
    # 혹시 다른 포맷이 섞여 있으면 mixed로 한 번 더 시도
    missing = df["published_utc"].isna()
    if missing.any():
        df.loc[missing, "published_utc"] = pd.to_datetime(
            df.loc[missing, "published_raw"], format="mixed",
            utc=True, errors="coerce",
        )
    df = df.dropna(subset=["published_utc"]).reset_index(drop=True)
    df["published_kst"] = df["published_utc"].dt.tz_convert(KST)
    df["date"] = df["published_kst"].dt.date
    # 기간 필터
    start = pd.to_datetime(start_date).date()
    end   = pd.to_datetime(end_date).date()
    df = df[(df["date"] >= start) & (df["date"] <= end)].reset_index(drop=True)
    return df

def ensure_normalized(df):
    """이미 정규화된 캐시(published_kst 컬럼 있음)는 그대로, 아니면 normalize 호출."""
    if 'published_kst' in df.columns:
        df = df.copy()
        df['published_kst'] = pd.to_datetime(df['published_kst'], errors='coerce')
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.date
        start = pd.to_datetime(START_DATE).date()
        end   = pd.to_datetime(END_DATE).date()
        df = df[(df['date'] >= start) & (df['date'] <= end)].reset_index(drop=True)
        return df
    return normalize_timestamps(df, START_DATE, END_DATE)

df_kr = ensure_normalized(df_kr_raw)
df_us = ensure_normalized(df_us_raw)

print(f"한국 뉴스 (KST 기간 내): {len(df_kr)}건")
print(f"미국 뉴스 (KST 기간 내): {len(df_us)}건")
print(f"한국 일별 평균: {len(df_kr) / max(1, df_kr['date'].nunique()):.1f}건")
print(f"미국 일별 평균: {len(df_us) / max(1, df_us['date'].nunique()):.1f}건")


In [ ]:
# 캐시 저장 (재실행 시 RSS 재호출 불필요)
df_kr.to_csv(DATA_DIR / "news_kr.csv", index=False)
df_us.to_csv(DATA_DIR / "news_us.csv", index=False)
print(f"저장: {DATA_DIR/'news_kr.csv'}, {DATA_DIR/'news_us.csv'}")


## 3. 유가 데이터 수집 (yfinance)

- WTI: `CL=F`, Brent: `BZ=F`
- 종가(Close) 기준


In [ ]:
CACHE_OIL = DATA_DIR / "oil_prices.csv"
if USE_CACHE and CACHE_OIL.exists():
    oil = pd.read_csv(CACHE_OIL, index_col=0, parse_dates=True)
    oil.index = pd.to_datetime(oil.index).tz_localize(None) if oil.index.tz else pd.to_datetime(oil.index)
    print(f"📦 캐시 로드: {CACHE_OIL.name} — {len(oil)}일")
else:
    # yfinance end 파라미터는 exclusive이므로 +1일
    end_inclusive = (pd.to_datetime(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    wti = yf.download("CL=F", start=START_DATE, end=end_inclusive, progress=False, auto_adjust=False)
    brent = yf.download("BZ=F", start=START_DATE, end=end_inclusive, progress=False, auto_adjust=False)
    oil = pd.DataFrame({
        "WTI":   wti["Close"].squeeze() if "Close" in wti.columns else np.nan,
        "Brent": brent["Close"].squeeze() if "Close" in brent.columns else np.nan,
    })
    oil.index = pd.to_datetime(oil.index).tz_localize(None)
    print(f"유가 데이터: {oil.shape[0]} 일치 (거래일 기준)")
oil.tail()


In [ ]:
oil.to_csv(DATA_DIR / "oil_prices.csv")
print(f"저장: {DATA_DIR/'oil_prices.csv'}")


## 4. 일별 키워드 빈도 매트릭스

각 키워드 그룹별로 헤드라인에 별칭 중 하나라도 포함되면 +1.


In [ ]:
def daily_keyword_counts(df, keywords_dict):
    """date x keyword 빈도 매트릭스 + total_headlines 컬럼 반환."""
    if df.empty:
        return pd.DataFrame()
    df = df.copy()
    for kw_name, aliases in keywords_dict.items():
        pattern = "|".join([re.escape(a) for a in aliases])
        df[kw_name] = df["title"].str.contains(pattern, case=False, na=False, regex=True).astype(int)
    keyword_cols = list(keywords_dict.keys())
    daily = df.groupby("date")[keyword_cols].sum()
    daily["total_headlines"] = df.groupby("date").size()
    # 전체 기간 인덱스로 재정렬 (누락된 날은 0)
    full_idx = pd.date_range(START_DATE, END_DATE, freq="D").date
    daily = daily.reindex(full_idx, fill_value=0)
    daily.index.name = "date"
    return daily

daily_kr = daily_keyword_counts(df_kr, KEYWORDS_KR)
daily_us = daily_keyword_counts(df_us, KEYWORDS_US)

print("=== 한국 일별 키워드 빈도 (head) ===")
display(daily_kr.head())
print("\n=== 미국 일별 키워드 빈도 (head) ===")
display(daily_us.head())


In [ ]:
# 캐시 저장
daily_kr.to_csv(DATA_DIR / "daily_keyword_freq_kr.csv")
daily_us.to_csv(DATA_DIR / "daily_keyword_freq_us.csv")

# 키워드별 전 기간 합계
print("=== 한국 키워드 누적 빈도 ===")
print(daily_kr.drop(columns=["total_headlines"]).sum().sort_values(ascending=False))
print("\n=== 미국 키워드 누적 빈도 ===")
print(daily_us.drop(columns=["total_headlines"]).sum().sort_values(ascending=False))


## 4.5 정규화 메트릭 추가 (v5)

raw 카운트만 보면 **헤드라인이 많은 날은 자동으로 키워드 카운트도 커짐** → 노이즈.
2가지 메트릭으로 비교:

| 메트릭 | 수식 | 의미 |
|--------|------|------|
| **raw** | `keyword` | 원본 카운트 — "총 보도량" 신호 |
| **ratio** | `keyword / total × 100` | 그날 분쟁 담론 중 비중 (%) |

> 참고: 이전 v3/v4에서 시도했던 laplace 평활은 저-count 키워드(시아 벨트 등 0~10건)에서
> `(k+1)/(total+10)` 분자가 prior에 지배돼 `1/total`과 거의 같아지면서 가짜 음의 상관을 만들어내는
> 인공물이 있어 제외함.


In [ ]:
def add_normalized_metrics(daily_df, keyword_cols):
    """daily_df에 ratio 컬럼 추가 (in-place). raw는 이미 keyword 컬럼."""
    total = daily_df["total_headlines"]
    for kw in keyword_cols:
        # 단순 비율 (총 헤드라인 대비 %)
        daily_df[f"{kw}__ratio"] = (
            daily_df[kw] / total.replace(0, np.nan) * 100
        )
    return daily_df

kw_cols_kr = list(KEYWORDS_KR.keys())
kw_cols_us = list(KEYWORDS_US.keys())

daily_kr = add_normalized_metrics(daily_kr, kw_cols_kr)
daily_us = add_normalized_metrics(daily_us, kw_cols_us)

# 샘플: 호르무즈 해협의 raw vs ratio (3월 1주차)
sample_cols = ["total_headlines", "호르무즈 해협", "호르무즈 해협__ratio"]
sample_idx = pd.date_range("2026-03-01", "2026-03-07", freq="D").date
print("=== 호르무즈 해협 — raw vs ratio (2026-03-01 ~ 03-07) ===")
display(daily_kr.loc[sample_idx, sample_cols].round(3))


## 5. 상관분석 (Pearson / Spearman + 시차)

- 유가는 거래일만 있으므로 **D 빈도 forward-fill** (주말은 직전 거래일 종가 유지)
- 시차(lag) `-7` ~ `+7` 일:
  - `lag = -k`: 뉴스가 유가보다 k일 **선행** (뉴스→유가)
  - `lag = +k`: 뉴스가 유가보다 k일 **후행** (유가→뉴스)


In [ ]:
# 유가 일별 리샘플링 (forward-fill)
oil_daily = oil.reindex(pd.date_range(START_DATE, END_DATE, freq="D")).ffill()
oil_daily.index = oil_daily.index.date
oil_daily.index.name = "date"

# 빈도 매트릭스에 유가 컬럼 붙이기
merged_kr = daily_kr.join(oil_daily)
merged_us = daily_us.join(oil_daily)
merged_kr.head()


In [ ]:
def lagged_correlation(df_daily, keyword_cols, oil_col, lags=range(-7, 8), suffix=""):
    """각 키워드와 oil_col 간 시차별 Pearson/Spearman 상관계수.

    suffix: '' (raw), '__ratio', '__laplace', '__smooth' 중 하나를 지정해서
    동일 함수로 모든 메트릭 처리 가능.
    """
    rows = []
    for kw in keyword_cols:
        col = f"{kw}{suffix}"
        if col not in df_daily.columns:
            continue
        for lag in lags:
            x = df_daily[col].shift(-lag)
            y = df_daily[oil_col]
            valid = pd.concat([x, y], axis=1).dropna()
            if len(valid) < 15 or valid.iloc[:, 0].std() == 0:
                continue
            pr, pp = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
            sr, sp = stats.spearmanr(valid.iloc[:, 0], valid.iloc[:, 1])
            rows.append({"keyword": kw, "lag": lag,
                         "pearson_r": pr, "pearson_p": pp,
                         "spearman_r": sr, "spearman_p": sp,
                         "n": len(valid)})
    return pd.DataFrame(rows)

# raw (v2 호환)
corr_kr_brent = lagged_correlation(merged_kr, kw_cols_kr, "Brent")
corr_kr_wti   = lagged_correlation(merged_kr, kw_cols_kr, "WTI")
corr_us_brent = lagged_correlation(merged_us, kw_cols_us, "Brent")
corr_us_wti   = lagged_correlation(merged_us, kw_cols_us, "WTI")

print(f"한국 vs Brent: {len(corr_kr_brent)} 행")
print(f"한국 vs WTI:   {len(corr_kr_wti)} 행")
print(f"미국 vs Brent: {len(corr_us_brent)} 행")
print(f"미국 vs WTI:   {len(corr_us_wti)} 행")


In [ ]:
def best_lag_summary(corr_df, metric="pearson_r"):
    """키워드별 |상관계수|가 최대인 lag와 그 값을 표시."""
    rows = []
    for kw, g in corr_df.groupby("keyword"):
        idx = g[metric].abs().idxmax()
        rows.append(g.loc[idx])
    return pd.DataFrame(rows).sort_values(metric, key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

print("=== [한국 뉴스 ↔ Brent] 키워드별 최강 시차 상관 (Pearson) ===")
display(best_lag_summary(corr_kr_brent, "pearson_r"))

print("\n=== [미국 뉴스 ↔ Brent] 키워드별 최강 시차 상관 (Pearson) ===")
display(best_lag_summary(corr_us_brent, "pearson_r"))


## 5.5 메트릭별 시차 상관 (v3)

raw 외에 ratio / laplace / smooth 각각에 대해서도 lag −7~+7 상관분석을 돌려서,
키워드별로 어떤 메트릭이 가장 강한 신호를 주는지 비교.


In [ ]:
METRICS = [("", "raw"), ("__ratio", "ratio")]

corr_results = {}  # key: (country, oil, metric_name) -> DataFrame
for country, merged, kw_cols in [("KR", merged_kr, kw_cols_kr),
                                  ("US", merged_us, kw_cols_us)]:
    for oil_col in ("Brent", "WTI"):
        for suffix, mname in METRICS:
            key = (country, oil_col, mname)
            corr_results[key] = lagged_correlation(merged, kw_cols, oil_col, suffix=suffix)

print("계산 완료:")
for k, v in list(corr_results.items())[:4]:
    print(f"  {k}: {len(v)} 행")
print(f"  ... 총 {len(corr_results)} 개의 (국가, 유가, 메트릭) 조합")


In [ ]:
def metric_compare_table(country, oil_col, kw_cols, corr_results):
    """키워드 × 메트릭 → 최강 r 및 lag 표."""
    rows = []
    for kw in kw_cols:
        row = {"keyword": kw}
        for _, mname in METRICS:
            df = corr_results[(country, oil_col, mname)]
            g = df[df["keyword"] == kw]
            if len(g) == 0:
                row[f"{mname}_r"] = np.nan
                row[f"{mname}_lag"] = np.nan
                continue
            idx = g["pearson_r"].abs().idxmax()
            row[f"{mname}_r"]   = g.loc[idx, "pearson_r"]
            row[f"{mname}_lag"] = int(g.loc[idx, "lag"])
        rows.append(row)
    df = pd.DataFrame(rows)
    # raw |r| 기준 정렬
    df["__sort"] = df["raw_r"].abs()
    df = df.sort_values("__sort", ascending=False).drop(columns="__sort").reset_index(drop=True)
    return df

print("=== [한국 뉴스 ↔ Brent] 메트릭별 최강 상관 비교 ===")
table_kr = metric_compare_table("KR", "Brent", kw_cols_kr, corr_results)
display(table_kr.round(3))

print("\n=== [미국 뉴스 ↔ Brent] 메트릭별 최강 상관 비교 ===")
table_us = metric_compare_table("US", "Brent", kw_cols_us, corr_results)
display(table_us.round(3))


In [ ]:
def plot_dual_axis_timeseries(daily_df, oil_df, country_label, oil_col="Brent"):
    fig, ax1 = plt.subplots(figsize=(14, 5))
    kw_cols = [c for c in daily_df.columns if c not in ("total_headlines", "WTI", "Brent")]
    total_kw = daily_df[kw_cols].sum(axis=1)
    ax1.fill_between(pd.to_datetime(daily_df.index), total_kw,
                     alpha=0.35, color=COLOR["cyan"], label="키워드 총 빈도",
                     edgecolor="none")
    ax1.plot(pd.to_datetime(daily_df.index), total_kw,
             color=COLOR["cyan"], linewidth=1.2, alpha=0.9)
    ax1.set_ylabel("키워드 총 빈도 (일별)", color=COLOR["cyan"])
    ax1.tick_params(axis="y", labelcolor=COLOR["cyan"])

    ax2 = ax1.twinx()
    ax2.plot(pd.to_datetime(oil_df.index), oil_df[oil_col],
             color=COLOR["amber"], linewidth=2, label=oil_col)
    ax2.set_ylabel(f"{oil_col} ($/배럴)", color=COLOR["amber"])
    ax2.tick_params(axis="y", labelcolor=COLOR["amber"])
    ax2.grid(False)  # 두 번째 axis는 grid 끔
    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color(COLOR["g_700"])

    ax1.set_title(f"[{country_label}] 키워드 빈도 vs {oil_col}", pad=14)
    apply_korean_font(ax1); apply_korean_font(ax2)
    fig.tight_layout()
    plt.show()

plot_dual_axis_timeseries(merged_kr, oil_daily, "한국 뉴스", "Brent")
plot_dual_axis_timeseries(merged_us, oil_daily, "미국 뉴스", "Brent")


### 6-2. 상관계수 히트맵 (키워드 × 시차)


In [ ]:
def plot_corr_heatmap(corr_df, title, metric="pearson_r"):
    pivot = corr_df.pivot(index="keyword", columns="lag", values=metric)
    fig, ax = plt.subplots(figsize=(12, max(4, 0.45 * len(pivot))))
    sns.heatmap(pivot, cmap=CMAP_DIV, center=0, annot=True, fmt=".2f",
                vmin=-0.7, vmax=0.7,
                annot_kws={"size": 9, "weight": "bold"},
                linewidths=1, linecolor=COLOR["bg"],
                cbar_kws={"label": metric, "shrink": 0.7}, ax=ax)
    ax.set_title(title, pad=14)
    ax.set_xlabel("lag (일)   ·   음수 = 뉴스가 유가보다 선행", labelpad=10)
    ax.set_ylabel("")
    # Colorbar 스타일
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(color=COLOR["g_400"])
    cbar.outline.set_edgecolor(COLOR["g_700"])
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=COLOR["fg"])
    apply_korean_font(ax)
    plt.tight_layout()
    plt.show()

plot_corr_heatmap(corr_kr_brent, "[한국 뉴스 ↔ Brent] 시차 상관 히트맵")
plot_corr_heatmap(corr_us_brent, "[미국 뉴스 ↔ Brent] 시차 상관 히트맵")


### 6-3. 한국 vs 미국 비교 패널 (키워드별 최강 상관계수)


In [ ]:
def comparison_panel(corr_kr, corr_us, oil_col):
    best_kr = best_lag_summary(corr_kr, "pearson_r").set_index("keyword")["pearson_r"]
    best_us = best_lag_summary(corr_us, "pearson_r").set_index("keyword")["pearson_r"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 0.4 * max(len(best_kr), len(best_us)) + 2))

    # KR — color by sign (positive = cyan, negative = amber)
    kr_colors = [COLOR["cyan"] if v >= 0 else COLOR["amber"] for v in best_kr.values]
    axes[0].barh(best_kr.index, best_kr.values, color=kr_colors, edgecolor="none")
    axes[0].axvline(0, color=COLOR["g_500"], linewidth=0.8)
    axes[0].set_title(f"[한국 뉴스] {oil_col} 최강 상관", pad=12)
    axes[0].set_xlabel("Pearson r")
    axes[0].invert_yaxis()

    us_colors = [COLOR["cyan"] if v >= 0 else COLOR["amber"] for v in best_us.values]
    axes[1].barh(best_us.index, best_us.values, color=us_colors, edgecolor="none")
    axes[1].axvline(0, color=COLOR["g_500"], linewidth=0.8)
    axes[1].set_title(f"[미국 뉴스] {oil_col} 최강 상관", pad=12)
    axes[1].set_xlabel("Pearson r")
    axes[1].invert_yaxis()

    apply_korean_font(axes[0]); apply_korean_font(axes[1])
    plt.tight_layout()
    plt.show()

comparison_panel(corr_kr_brent, corr_us_brent, "Brent")


### 6-4. 최강 상관 키워드 산점도


In [ ]:
def scatter_top_keyword(merged_df, corr_df, oil_col, country_label):
    top = best_lag_summary(corr_df, "pearson_r").iloc[0]
    kw, lag = top["keyword"], int(top["lag"])
    x = merged_df[kw].shift(-lag)
    y = merged_df[oil_col]
    df = pd.concat([x, y], axis=1).dropna()
    df.columns = [f"{kw} (lag={lag})", oil_col]

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.regplot(
        data=df, x=df.columns[0], y=oil_col, ax=ax,
        scatter_kws={"alpha": 0.6, "color": COLOR["cyan"], "edgecolor": "none", "s": 32},
        line_kws={"color": COLOR["amber"], "linewidth": 2.2},
        ci=68,
    )
    r = top["pearson_r"]
    ax.set_title(
        f"[{country_label}] '{kw}' (lag={lag}) vs {oil_col}   "
        f"·   Pearson r = {r:.3f}",
        pad=14,
    )
    apply_korean_font(ax)
    plt.tight_layout()
    plt.show()

scatter_top_keyword(merged_kr, corr_kr_brent, "Brent", "한국 뉴스")
scatter_top_keyword(merged_us, corr_us_brent, "Brent", "미국 뉴스")


### 6-5. 워드클라우드 (참고용)


In [ ]:
try:
    from wordcloud import WordCloud
    # 한국어 워드클라우드 — 키워드 누적 빈도 기반
    freq_kr = daily_kr.drop(columns=["total_headlines"]).sum().to_dict()
    freq_us = daily_us.drop(columns=["total_headlines"]).sum().to_dict()

    # 한국어 폰트 경로 (시스템에 따라 조정)
    import matplotlib.font_manager as fm
    font_path = None
    for f in fm.fontManager.ttflist:
        if f.name in ("Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"):
            font_path = f.fname
            break
    if font_path is None:
        raise RuntimeError("한글 폰트 미발견")

    wc_kr = WordCloud(font_path=font_path, width=800, height=400,
                      background_color="white").generate_from_frequencies(freq_kr)
    wc_us = WordCloud(width=800, height=400,
                      background_color="white").generate_from_frequencies(freq_us)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].imshow(wc_kr); axes[0].axis("off"); axes[0].set_title("한국 뉴스 키워드")
    axes[1].imshow(wc_us); axes[1].axis("off"); axes[1].set_title("US News Keywords")
    apply_korean_font(axes[0]); apply_korean_font(axes[1])
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"워드클라우드 생성 실패: {e}")


### 6-6. 메트릭 비교 히트맵 (v3) — raw vs ratio vs laplace vs smooth

각 키워드의 4가지 메트릭에서 나온 최강 |Pearson r| 비교.
**raw가 강한데 ratio/laplace에서 약해지면 → 그 신호는 보도량 자체에 휘둘린 것**.
**ratio/laplace에서도 강하면 → 어젠다 비중 자체가 유가와 동조**.


In [ ]:
def plot_metric_comparison_heatmap(country, oil_col, kw_cols, corr_results, country_label):
    rows = []
    for kw in kw_cols:
        for _, mname in METRICS:
            df = corr_results[(country, oil_col, mname)]
            g = df[df["keyword"] == kw]
            if len(g) == 0:
                continue
            idx = g["pearson_r"].abs().idxmax()
            rows.append({"keyword": kw, "metric": mname,
                         "best_r": g.loc[idx, "pearson_r"],
                         "best_lag": g.loc[idx, "lag"]})
    long = pd.DataFrame(rows)
    pivot = long.pivot(index="keyword", columns="metric", values="best_r")
    pivot = pivot[["raw", "ratio"]]
    pivot = pivot.reindex(pivot["raw"].abs().sort_values(ascending=False).index)

    fig, ax = plt.subplots(figsize=(9, 0.5 * len(pivot) + 2))
    sns.heatmap(pivot, cmap=CMAP_DIV, center=0, annot=True, fmt=".3f",
                vmin=-0.7, vmax=0.7,
                annot_kws={"size": 10, "weight": "bold"},
                linewidths=1, linecolor=COLOR["bg"],
                cbar_kws={"label": "best |Pearson r|", "shrink": 0.7}, ax=ax)
    ax.set_title(f"[{country_label}] 키워드 × 메트릭 — {oil_col} 최강 상관", pad=14)
    ax.set_ylabel("")
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(color=COLOR["g_400"])
    cbar.outline.set_edgecolor(COLOR["g_700"])
    plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=COLOR["fg"])
    apply_korean_font(ax)
    plt.tight_layout()
    plt.show()
    return pivot

print("=== [한국 뉴스] 메트릭 비교 히트맵 ===")
pivot_kr = plot_metric_comparison_heatmap("KR", "Brent", kw_cols_kr, corr_results, "한국 뉴스")
print("\n=== [미국 뉴스] 메트릭 비교 히트맵 ===")
pivot_us = plot_metric_comparison_heatmap("US", "Brent", kw_cols_us, corr_results, "미국 뉴스")


### 6-7. Top 키워드 안정성 플롯 — raw vs ratio

호르무즈 해협 / Strait of Hormuz에 대해 raw와 ratio가 얼마나 같이 움직이는지 확인.


In [ ]:
def plot_metric_stability(daily_df, oil_df, kw, country_label, oil_col="Brent"):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    suffixes = [("",        "raw count",  COLOR["cyan"]),
                ("__ratio", "ratio (%)",  COLOR["cyan_lt"])]
    ax = axes[0]
    for suf, label, color in suffixes:
        col = f"{kw}{suf}"
        if col not in daily_df.columns:
            continue
        s = daily_df[col]
        s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
        ax.plot(pd.to_datetime(daily_df.index), s_norm, label=label,
                color=color, alpha=0.9, linewidth=1.6)
    ax.set_title(f"[{country_label}] '{kw}' — raw vs ratio (각자 min-max 정규화)", pad=12)
    ax.set_ylabel("정규화된 값 (0~1)")
    ax.legend(loc="upper left", framealpha=0.9)
    apply_korean_font(ax)

    ax2 = axes[1]
    ax2.plot(pd.to_datetime(oil_df.index), oil_df[oil_col],
             color=COLOR["amber"], linewidth=2)
    ax2.set_ylabel(f"{oil_col} ($/배럴)", color=COLOR["amber"])
    ax2.tick_params(axis="y", labelcolor=COLOR["amber"])
    ax2.set_title(f"{oil_col} 가격 (참조용)", pad=12)
    apply_korean_font(ax2)
    plt.tight_layout()
    plt.show()

plot_metric_stability(merged_kr, oil_daily, "호르무즈 해협", "한국 뉴스", "Brent")
plot_metric_stability(merged_us, oil_daily, "Strait of Hormuz", "미국 뉴스", "Brent")


## 6-8. WTI 분석 추가 (v4)

지금까지는 Brent만 시각화했지만, WTI도 같이 분석. 두 벤치마크는 거의 동조 (자체 상관 r≈0.98)
하지만 키워드별로 미세한 차이가 있을 수 있음.


In [ ]:
# 두 유가를 한 차트에 동시 표시 (키워드 빈도 + Brent + WTI)
def plot_dual_axis_both_oils(daily_df, oil_df, country_label):
    fig, ax1 = plt.subplots(figsize=(14, 5))
    kw_cols = [c for c in daily_df.columns if c not in ("total_headlines", "WTI", "Brent")
               and not any(s in c for s in ("__ratio", "__laplace", "__smooth"))]
    total_kw = daily_df[kw_cols].sum(axis=1)
    ax1.fill_between(pd.to_datetime(daily_df.index), total_kw,
                     alpha=0.35, color=COLOR["cyan"], label="키워드 총 빈도",
                     edgecolor="none")
    ax1.plot(pd.to_datetime(daily_df.index), total_kw,
             color=COLOR["cyan"], linewidth=1.2, alpha=0.9)
    ax1.set_ylabel("키워드 총 빈도 (일별)", color=COLOR["cyan"])
    ax1.tick_params(axis="y", labelcolor=COLOR["cyan"])

    ax2 = ax1.twinx()
    ax2.plot(pd.to_datetime(oil_df.index), oil_df["Brent"],
             color=COLOR["amber"], linewidth=2.2, label="Brent")
    ax2.plot(pd.to_datetime(oil_df.index), oil_df["WTI"],
             color=COLOR["violet"], linewidth=1.8, linestyle="--", label="WTI")
    ax2.set_ylabel("유가 ($/배럴)", color=COLOR["fg"])
    ax2.tick_params(axis="y", labelcolor=COLOR["fg"])
    ax2.legend(loc="upper left", framealpha=0.9)
    ax2.grid(False)
    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color(COLOR["g_700"])

    ax1.set_title(f"[{country_label}] 키워드 빈도 vs Brent & WTI", pad=14)
    apply_korean_font(ax1); apply_korean_font(ax2)
    fig.tight_layout()
    plt.show()

plot_dual_axis_both_oils(merged_kr, oil_daily, "한국 뉴스")
plot_dual_axis_both_oils(merged_us, oil_daily, "미국 뉴스")


In [ ]:
# WTI 시차 상관 히트맵 (Brent와 동일 포맷)
plot_corr_heatmap(corr_kr_wti, "[한국 뉴스 ↔ WTI] 시차 상관 히트맵")
plot_corr_heatmap(corr_us_wti, "[미국 뉴스 ↔ WTI] 시차 상관 히트맵")


In [ ]:
# Brent vs WTI 키워드별 최강 상관 — 한 차트에 같이
def brent_vs_wti_panel(corr_brent, corr_wti, country_label):
    best_b = best_lag_summary(corr_brent, "pearson_r").set_index("keyword")["pearson_r"]
    best_w = best_lag_summary(corr_wti,   "pearson_r").set_index("keyword")["pearson_r"]
    common = best_b.index.intersection(best_w.index)
    df = pd.DataFrame({"Brent": best_b.loc[common], "WTI": best_w.loc[common]})
    df = df.reindex(df["Brent"].abs().sort_values(ascending=False).index)

    fig, ax = plt.subplots(figsize=(10, 0.5 * len(df) + 2))
    df.plot(kind="barh", ax=ax,
            color=[COLOR["amber"], COLOR["violet"]], width=0.78,
            edgecolor="none")
    ax.invert_yaxis()
    ax.axvline(0, color=COLOR["g_500"], linewidth=0.8)
    ax.set_xlabel("최강 Pearson r")
    ax.set_title(f"[{country_label}] 키워드별 최강 상관 — Brent vs WTI 비교", pad=14)
    ax.legend(framealpha=0.9)
    apply_korean_font(ax)
    plt.tight_layout()
    plt.show()
    return df

print("=== 한국 뉴스 ===")
df_kr = brent_vs_wti_panel(corr_kr_brent, corr_kr_wti, "한국 뉴스")
print("\n=== 미국 뉴스 ===")
df_us = brent_vs_wti_panel(corr_us_brent, corr_us_wti, "미국 뉴스")


In [ ]:
# 메트릭 비교 히트맵 — WTI 버전
print("=== [한국 뉴스] 메트릭 비교 히트맵 (WTI) ===")
pivot_kr_wti = plot_metric_comparison_heatmap("KR", "WTI", kw_cols_kr, corr_results, "한국 뉴스")
print("\n=== [미국 뉴스] 메트릭 비교 히트맵 (WTI) ===")
pivot_us_wti = plot_metric_comparison_heatmap("US", "WTI", kw_cols_us, corr_results, "미국 뉴스")


In [ ]:
# Brent 최강 vs WTI 최강을 직접 비교한 표
def brent_wti_comparison_table(country, kw_cols, corr_results):
    rows = []
    for kw in kw_cols:
        for metric_suf, mname in METRICS:
            b = corr_results[(country, "Brent", mname)]
            w = corr_results[(country, "WTI", mname)]
            gb = b[b["keyword"] == kw]
            gw = w[w["keyword"] == kw]
            if len(gb) == 0 or len(gw) == 0: continue
            idx_b = gb["pearson_r"].abs().idxmax()
            idx_w = gw["pearson_r"].abs().idxmax()
            rows.append({
                "keyword": kw, "metric": mname,
                "Brent_r": gb.loc[idx_b, "pearson_r"], "Brent_lag": int(gb.loc[idx_b, "lag"]),
                "WTI_r":   gw.loc[idx_w, "pearson_r"], "WTI_lag":   int(gw.loc[idx_w, "lag"]),
                "차이": gw.loc[idx_w, "pearson_r"] - gb.loc[idx_b, "pearson_r"],
            })
    return pd.DataFrame(rows)

print("=== [한국 뉴스] raw 메트릭에서 Brent vs WTI ===")
table_kr = brent_wti_comparison_table("KR", kw_cols_kr, corr_results)
display(table_kr[table_kr["metric"] == "raw"].drop(columns="metric").round(3))

print("\n=== [미국 뉴스] raw 메트릭에서 Brent vs WTI ===")
table_us = brent_wti_comparison_table("US", kw_cols_us, corr_results)
display(table_us[table_us["metric"] == "raw"].drop(columns="metric").round(3))


## 7. 해석 가이드 (분석 후 채우기)

- **양의 상관**(키워드 빈도↑ → 유가↑): 분쟁/리스크 보도가 활발할수록 유가 상승
- **음의 lag**(예: lag = -2): 뉴스가 2일 **먼저** 움직임 → 뉴스가 유가를 선행 신호
- **양의 lag**: 유가 변동 후 뉴스 보도가 따라옴 → 사후 보도 패턴

### 주의사항
- Google News RSS는 과거 데이터 완전성이 보장되지 않음. 일별 헤드라인 수가 적은 날은 `total_headlines`로 정규화한 빈도(`keyword / total_headlines`)도 함께 확인할 것.
- 상관관계 ≠ 인과관계. Granger causality 등 추가 검정 필요시 별도 모듈 작성.
